## Read & Write en Delta Lake
1. Escribir datos en Delta Lake(Managed Table)
2. Escribir datos en Delta Lake(External Table)
3. Leer datos desde Delta Lake (Table)
4. Leer datos desde Delta Lake (File)

In [0]:
%sql
CREATE SCHEMA If NOT EXISTS movie_demo
MANAGED LOCATION "abfss://demo@moviehistory22.dfs.core.windows.net/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movieschema = StructType( fields= [
    StructField('movieId', IntegerType(), False),
    StructField('title', StringType(), True),
    StructField('budget', DoubleType(), True),
    StructField('homePage', StringType(), True),
    StructField('overview', StringType(), True), 
    StructField('popularity', DoubleType(), True),
    StructField('yearReleaseDate', IntegerType(), True),
    StructField('releaseDate',  DateType(), True),
    StructField('revenue', DoubleType(), True),
    StructField('durationTime', IntegerType(), True),
    StructField('movieStatus', StringType(), True),
    StructField('tagline', StringType(), True),
    StructField('voteAverage', DoubleType(), True),
    StructField('voteCount', IntegerType(), True)
])

In [0]:
movie_df = spark.read \
            .option("header", True) \
            .schema(movieschema) \
            .csv("abfss://bronze@moviehistory22.dfs.core.windows.net/2024-12-30/movie.csv")

In [0]:
display(movie_df)

### EScribir datos en Delta Lake

In [0]:
# Paso1:  en formato tabla

movie_df.write.format("delta").mode("overwrite").saveAsTable("movie_demo.movies_managed")

In [0]:
%sql
SELECT * FROM movie_demo.movies_managed	

In [0]:
# Paso2: en formato archivo

movie_df.write.format("delta").mode("overwrite").save("abfss://demo@moviehistory22.dfs.core.windows.net/movies_external")

### Leer desde Deta Lake

Cargo los datos en fromato Delta en esta tabla:

In [0]:
%sql
CREATE TABLE movie_demo.movies_external
USING DELTA
LOCATION "abfss://demo@moviehistory22.dfs.core.windows.net/movies_external"

In [0]:
%sql
SELECT * FROM movie_demo.movies_external

Leer datos desde un archivo en formato delta a un DF

In [0]:
movies_external_df = spark.read.format("delta").load("abfss://demo@moviehistory22.dfs.core.windows.net/movies_external")

In [0]:
display(movies_external_df)

Tambien se puede particionar los datos... lo hago a partir de la escritura de una tabla administrada.

In [0]:
movie_df.write.format("delta").mode("overwrite").partitionBy("yearReleaseDate").saveAsTable("movie_demo.movies_partitioned")

Conocer la ubicacion de las tablas

In [0]:
%sql
DESC EXTENDED movie_demo.movies_partitioned

In [0]:
%sql
SHOW PARTITIONS movie_demo.movies_partitioned

## Update & Delete en Delta Lake
1. Update desde Delta Lake
2. Delete desde en Delta Lake

In [0]:
%sql
SELECT * FROM movie_demo.movies_managed

In [0]:
%sql
UPDATE movie_demo.movies_managed 
SET durationTime = 60 
WHERE yearReleaseDate = 2012;

In [0]:
%sql
SELECT * FROM movie_demo.movies_managed
WHERE yearReleaseDate = 2012

In [0]:
from delta.tables import *

# deltaTable = DeltaTable.forPath(spark, 'abfss://demo@moviehirstoy22.dfs.core.windows.net/movie_managed')
deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "yearReleaseDate = '2013'",
  set = { "durationTime": "100" }
)

In [0]:
%sql
SELECT * FROM movie_demo.movies_managed
WHERE yearReleaseDate = 2013

In [0]:
%sql
DELETE FROM movie_demo.movies_managed
WHERE yearReleaseDate = 2014

In [0]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, 'movie_demo.movies_managed')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("yearReleaseDate =2015")

## Merge / Upsert en Delta Lake

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movieschema = StructType( fields= [
    StructField('movieId', IntegerType(), False),
    StructField('title', StringType(), True),
    StructField('budget', DoubleType(), True),
    StructField('homePage', StringType(), True),
    StructField('overview', StringType(), True), 
    StructField('popularity', DoubleType(), True),
    StructField('yearReleaseDate', IntegerType(), True),
    StructField('releaseDate',  DateType(), True),
    StructField('revenue', DoubleType(), True),
    StructField('durationTime', IntegerType(), True),
    StructField('movieStatus', StringType(), True),
    StructField('tagline', StringType(), True),
    StructField('voteAverage', DoubleType(), True),
    StructField('voteCount', IntegerType(), True)
])

In [0]:
movie_day1_df = spark.read \
            .option("header", True) \
            .schema(movieschema) \
            .csv("abfss://bronze@moviehistory22.dfs.core.windows.net/2024-12-30/movie.csv") \
            .filter("yearReleaseDate < 2000") \
            .select("movieId", "title", "yearReleaseDate", "releaseDate", "durationTime")

In [0]:
display(movie_day1_df)

In [0]:
movie_day1_df.createOrReplaceTempView("movie_day1")

In [0]:
from pyspark.sql.functions import upper
movie_day2_df = spark.read \
            .option("header", True) \
            .schema(movieschema) \
            .csv("abfss://bronze@moviehistory22.dfs.core.windows.net/2024-12-30/movie.csv") \
            .filter("yearReleaseDate between 1998 and 2005") \
            .select("movieId", upper("title").alias("title"), "yearReleaseDate", "releaseDate", "durationTime")

In [0]:
display(movie_day2_df)

In [0]:
movie_day2_df.createOrReplaceTempView("movie_day2")

In [0]:
from pyspark.sql.functions import upper
movie_day3_df = spark.read \
            .option("header", True) \
            .schema(movieschema) \
            .csv("abfss://bronze@moviehistory22.dfs.core.windows.net/2024-12-30/movie.csv") \
            .filter("yearReleaseDate between 1983 and 1998 OR yearReleaseDate between 2006 and 2010") \
            .select("movieId", upper("title").alias("title"), "yearReleaseDate", "releaseDate", "durationTime")

In [0]:
display(movie_day3_df)

creo una tabla de destino

In [0]:
%sql
create table if not exists movie_demo.movies_merge(
    movieId INT, 
    title STRING,
    yearReleaseDate INT, 
    releaseDate DATE,
    durationTime INT,
    createdDate DATE,
    updatedDate DATE
)

#### DIA 1

In [0]:
%sql
MERGE INTO movie_demo.movies_merge as tgt
USING movie_day1 as src
ON tgt.movieId = src.movieId
WHEN MATCHED THEN
  UPDATE SET
    tgt.title = src.title,
    tgt.yearReleaseDate = src.yearReleaseDate,
    tgt.releaseDate = src.releaseDate,
    tgt.durationTime = src.durationTime,
    tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN 
  INSERT (tgt.movieId, tgt.title, tgt.yearReleaseDate, tgt.releaseDate, tgt.durationTime, tgt.createdDate)
  VALUES (src.movieId, src.title, src.yearReleaseDate, src.releaseDate, src.durationTime, current_timestamp)

In [0]:
%sql
Select * from movie_demo.movies_merge

#### DIA 2

In [0]:
%sql
MERGE INTO movie_demo.movies_merge as tgt
USING movie_day2 as src
ON tgt.movieId = src.movieId
WHEN MATCHED THEN
  UPDATE SET
    tgt.title = src.title,
    tgt.yearReleaseDate = src.yearReleaseDate,
    tgt.releaseDate = src.releaseDate,
    tgt.durationTime = src.durationTime,
    tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN 
  INSERT (tgt.movieId, tgt.title, tgt.yearReleaseDate, tgt.releaseDate, tgt.durationTime, tgt.createdDate)
  VALUES (src.movieId, src.title, src.yearReleaseDate, src.releaseDate, src.durationTime, current_timestamp)

In [0]:
%sql
Select * from movie_demo.movies_merge

#### DIA 3

In [0]:
from delta.tables import *

deltaTablePeople = DeltaTable.forName(spark, 'movie_demo.movies_merge')

deltaTablePeople.alias('tgt') \
  .merge(
    movie_day3_df.alias('src'),
    'tgt.movieId = src.movieId'
  ) \
  .whenMatchedUpdate(set =
    {
        "tgt.title":"src.title",
        "tgt.yearReleaseDate":"src.yearReleaseDate",
        "tgt.releaseDate": "src.releaseDate",
        "tgt.durationTime": "src.durationTime",
        "tgt.updatedDate": "current_timestamp()"
    }
  ) \
  .whenNotMatchedInsert(values =
    {
        "tgt.movieId":"src.movieId",
        "tgt.title":"src.title",
        "tgt.yearReleaseDate":"src.yearReleaseDate",
        "tgt.releaseDate": "src.releaseDate",
        "tgt.durationTime": "src.durationTime",
        "tgt.createdDate": "current_timestamp()"
    }
  ) \
  .execute()

In [0]:
%sql
Select * from movie_demo.movies_merge

## History, Time Travel y Vacuum
1. Historia y Control de Versiones
2. Viaje en el Tiempo
3. Vacio

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

vero que paso en la primera version

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge VERSION AS OF 1

tambien se puede por timestamp

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge TIMESTAMP AS OF '2026-03-28T02:48:37.000+00:00'

In [0]:
df = spark.read.format("delta").option("timestampAsOf","2026-03-28T02:48:37.000+00:00").table("movie_demo.movies_merge")

In [0]:
display(df)

Eliminar historias

In [0]:
%sql
VACUUM movie_demo.movies_merge

databricks las mantiene por 7 dias.. por lo tanto , puedo ejeuctar

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge TIMESTAMP AS OF '2026-03-28T02:48:37.000+00:00'

se puede forzar la eliminacion

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM movie_demo.movies_merge RETAIN 0 HOURS;

CASO PARTICULAR

In [0]:
%sql
select * from movie_demo.movies_merge

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

In [0]:
%sql
DELETE FROM movie_demo.movies_merge
WHERE yearReleaseDate = 2004

In [0]:
%sql
select * from movie_demo.movies_merge

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

In [0]:
%sql
merge into movie_demo.movies_merge tgt
using movie_demo.movies_merge version as of 9 src
on tgt.movieId = src.movieId
When not matched then 
  insert *

In [0]:
%sql
DESC HISTORY movie_demo.movies_merge

# Transaction Log en Delta Lake

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movies_log(
  movie_Id INT, 
  title STRING,
  yearreleaseDate INT,
  releaseDate DATE, 
  durationTime INT, 
  createdDate DATE, 
  updatedDate DATE
)
USING DELTA

In [0]:
%sql
DESC HISTORY movie_demo.movies_log

In [0]:
%sql
DESC EXTENDED movie_demo.movies_log

In [0]:
%sql
insert into movie_demo.movies_log
Select * from movie_demo.movies_merge
where movieId = 125537


In [0]:
%sql
DESC HISTORY movie_demo.movies_log

In [0]:
%sql
insert into movie_demo.movies_log
Select * from movie_demo.movies_merge
where movieId = 133575

In [0]:
%sql
DESC HISTORY movie_demo.movies_log

In [0]:
%sql
DELETE FROM movie_demo.movies_log
where movie_Id = 125537

In [0]:
%sql
DESC HISTORY movie_demo.movies_log

INsertar registros de manera individual a partir de una lista

In [0]:
list = [118452, 124606, 125052, 125123, 125263, 125537, 126141, 133575, 142132, 146269, 157185]
for movieId in list:
  spark.sql(f"""INSERT INTO movie_demo.movies_log
            SELECT * FROM movie_demo.movies_merge
            where movieId = {movieId}""")

In [0]:
%sql
INSERT INTO movie_demo.movies_log
SELECT * FROM movie_demo.movies_merge

## Convertir formato "Parquet" a "Delta"

creo una tabla external desde un parquet y luego la convierto en delta

In [0]:
%sql
CREATE TABLE IF NOT EXISTS movie_demo.movies_convert_to_delta(
  movieId INT, 
  title STRING,
  yearReleaseDate INT,
  releaseDate DATE, 
  durationTime INT, 
  createdDate DATE, 
  updatedDate DATE
)
USING PARQUET
LOCATION 'abfss://demo@moviehistory22.dfs.core.windows.net/movies_convert_to_delta'

In [0]:
%sql
INSERT INTO movie_demo.movies_convert_to_delta
SELECT * FROM movie_demo.movies_merge

In [0]:
%sql
CONVERT TO DELTA movie_demo.movies_convert_to_delta

de directorio parquet a tabla delta:

In [0]:
df = spark.table("movie_demo.movies_convert_to_delta")
display(df)

In [0]:
df.write.format("parquet").save("abfss://demo@moviehistory22.dfs.core.windows.net/movies_convert_to_delta_new")

In [0]:
%sql
CONVERT TO DELTA parquet.`abfss://demo@moviehistory22.dfs.core.windows.net/movies_convert_to_delta_new`
    


Ojo arriba con las comillas... son ` (invertidas) sino falla.

In [0]:
%sql
SHOW GRANTS ON SCHEMA databricks_course_ws.movie_silver;